# EgoBlur -- parallel egocentric-video anonymizer (Colab T4)

Anonymize one video with **Meta EgoBlur (gen2)** -- faces (+ optional licence plates) -- on a single T4 with **both speedups stacked**, stitch the result into one clip, and report the numbers that feed the thesis metrics (detections, face/plate **coverage**, FPS, peak VRAM).

**Design choices baked in** (from earlier benchmarking):
- **N worker processes** (default 2), each its own CUDA context, each over an equal frame range -- fills the shared GPU's idle gaps.
- **Per-process batching** (default 4, the measured single-process sweet spot) through the detectron2 model.
- **Native resolution** (fit to the data, no rescaling) and **FP32**.
- Each worker **self-checks** the batched path against EgoBlur's trusted per-frame `run()` at startup and falls back to per-frame on any mismatch, so the M1 face numbers stay trustworthy.

> First: `Runtime > Change runtime type > T4 GPU`.
> EgoBlur weights are licence-gated (EgoBlur repo / Project Aria).
> **Never commit any unblurred frame or identifying media** (`code/README.md`). Run the cells top to bottom.

## Step 1 -- runtime check

In [ ]:
# Confirm a T4 GPU is attached.
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    print("GPU:", name, "| total VRAM (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    if "T4" not in name:
        print("WARNING: not a T4 -- the thesis targets a single T4, numbers won't transfer.")
else:
    print("No GPU. Runtime > Change runtime type > T4 GPU.")

## Step 2 -- install + clone EgoBlur

In [ ]:
# OpenCV (+ matplotlib for previews) and the EgoBlur repo (its gen2 predictor +
# vendored detectron2 patch). The .jit models are self-contained TorchScript, so
# no detectron2 install is needed.
!pip install -q opencv-python-headless matplotlib
import os
if not os.path.isdir("/content/EgoBlur"):
    import subprocess
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/facebookresearch/EgoBlur.git", "/content/EgoBlur"],
                   check=True)
print("install + clone done")

## Step 3 -- EgoBlur weights (Google Drive)
The two `.jit` files are licence-gated and not redistributable. Put them in your Drive and point to them; you need at least the face model.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Edit to your files. Set LP_MODEL = None if you only have the face model.
FACE_MODEL = "/content/drive/MyDrive/Licencjat/ego_blur_face_gen2.jit"
LP_MODEL   = "/content/drive/MyDrive/Licencjat/ego_blur_lp_gen2.jit"

import os
for tag, p in [("FACE", FACE_MODEL), ("LP", LP_MODEL)]:
    if p and os.path.exists(p):
        print(tag, "->", round(os.path.getsize(p) / 1e6, 1), "MB")
    else:
        print(tag, "-> MISSING:", p)
assert FACE_MODEL or LP_MODEL, "Set FACE_MODEL and/or LP_MODEL to your .jit path(s)."

## Step 4 -- your video + orientation
Probe the clip, then use the preview to choose the upright orientation.

In [ ]:
import cv2
VIDEO_PATH = "/content/drive/MyDrive/Licencjat/ego_data/stereo_right.mp4"   # <-- edit

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open {VIDEO_PATH}"
fps_in = cap.get(cv2.CAP_PROP_FPS) or 30
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
N_TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
print(f"{VIDEO_PATH}\n{W}x{H} @ {fps_in:.0f} fps | ~{N_TOTAL} frames")

In [ ]:
# EgoBlur needs UPRIGHT faces. Pick the ROTATE value (you set it in Step 6)
# that makes faces upright in this preview.
import matplotlib.pyplot as plt
cap = cv2.VideoCapture(VIDEO_PATH); ok, fr0 = cap.read(); cap.release()
opts = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180,
        270: cv2.ROTATE_90_COUNTERCLOCKWISE}
fig, ax = plt.subplots(1, 4, figsize=(16, 5))
for a, (deg, flag) in zip(ax, opts.items()):
    img = fr0 if flag is None else cv2.rotate(fr0, flag)
    a.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); a.set_title(f"ROTATE = {deg}"); a.axis("off")
plt.tight_layout(); plt.show()

## Step 5 -- write the worker
Writes `egoblur_worker.py`: one process that batches its frame range `[start, end)` through the model and writes a segment + stats. Several run at once in Step 6. Just run the cell.

In [ ]:
%%writefile egoblur_worker.py
"""One EgoBlur worker: anonymize frames [start, end) of a video, write a segment.
Runs as its OWN process (own CUDA context) so several can share one GPU. Inside
the process it BATCHES frames (default 4) through the detectron2 model. FP32,
native resolution. A startup self-check compares the batched output to the
trusted per-frame path (EgoblurDetector.run); on any mismatch it falls back to
per-frame, so the M1 face numbers stay trustworthy."""
import argparse, sys, json, time
import numpy as np, cv2, torch
import torchvision


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--egoblur-repo", default="/content/EgoBlur")
    ap.add_argument("--face-model", default="")
    ap.add_argument("--lp-model", default="")
    ap.add_argument("--video", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--stats", required=True)
    ap.add_argument("--progress", default="")
    ap.add_argument("--start", type=int, required=True)
    ap.add_argument("--end", type=int, required=True)            # exclusive
    ap.add_argument("--min-size", type=int, required=True)
    ap.add_argument("--max-size", type=int, required=True)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--face-thr", type=float, default=0.674)
    ap.add_argument("--lp-thr", type=float, default=0.745)
    ap.add_argument("--rotate", type=int, default=0)
    ap.add_argument("--redact", default="blur")                  # "blur" or "blue"
    a = ap.parse_args()

    if a.egoblur_repo not in sys.path:
        sys.path.insert(0, a.egoblur_repo)
    from gen2.script.predictor import ClassID, EgoblurDetector, PATCH_INSTANCES_FIELDS
    from gen2.script.detectron2.export.torchscript_patch import patch_instances

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    resize = {"min_size_test": a.min_size, "max_size_test": a.max_size}
    cfg, thrs = [], []
    if a.face_model:
        cfg.append(("face", EgoblurDetector(
            model_path=a.face_model, device=dev, detection_class=ClassID.FACE,
            score_threshold=a.face_thr, nms_iou_threshold=0.3, resize_aug=resize)))
        thrs.append(a.face_thr)
    if a.lp_model:
        cfg.append(("licence plate", EgoblurDetector(
            model_path=a.lp_model, device=dev, detection_class=ClassID.LICENSE_PLATE,
            score_threshold=a.lp_thr, nms_iou_threshold=0.3, resize_aug=resize)))
        thrs.append(a.lp_thr)

    def script_model(det):
        for v in vars(det).values():
            if isinstance(v, torch.jit.RecursiveScriptModule):
                return v
        raise RuntimeError("scripted model not found on EgoblurDetector")
    models = [script_model(d) for _, d in cfg]

    ROT = {0: None, 90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180,
           270: cv2.ROTATE_90_COUNTERCLOCKWISE}
    rflag = ROT[a.rotate]

    def resize_native(fr):
        h, w = fr.shape[:2]
        scale = a.min_size / min(h, w)
        if max(h, w) * scale > a.max_size:
            scale = a.max_size / max(h, w)
        nh, nw = max(1, int(round(h * scale))), max(1, int(round(w * scale)))
        return cv2.resize(fr, (nw, nh), interpolation=cv2.INTER_LINEAR), (w / nw, h / nh)

    def detect_one(fr):                          # trusted per-frame path
        t = torch.from_numpy(np.ascontiguousarray(fr.transpose(2, 0, 1))).to(dev)
        ds = []
        for label, det in cfg:
            o = det.run(t)
            if o and isinstance(o[0], list) and (len(o[0]) == 0 or isinstance(o[0][0], list)):
                o = o[0]
            for b in o:
                if len(b) < 4:
                    continue
                x1, y1, x2, y2 = (int(round(v)) for v in b[:4])
                ds.append(((x1, y1, x2, y2), label))
        return ds

    def detect_batch(frames):                    # batched path (one inference call / model)
        results = [[] for _ in frames]
        for (label, _det), model, thr in zip(cfg, models, thrs):
            batch, scales = [], []
            for fr in frames:
                rz, sc = resize_native(fr)
                batch.append({"image": torch.from_numpy(
                    np.ascontiguousarray(rz.transpose(2, 0, 1))).to(dev)})
                scales.append(sc)
            for i, inst in enumerate(model.inference(batch, do_postprocess=False)):
                sx, sy = scales[i]
                pb = inst.pred_boxes
                boxes = pb.tensor if hasattr(pb, "tensor") else pb
                scores = inst.scores
                keep = scores >= thr
                boxes, scores = boxes[keep], scores[keep]
                if boxes.shape[0]:
                    boxes = boxes[torchvision.ops.nms(boxes, scores, 0.3)]
                for b in boxes.tolist():
                    results[i].append(((int(round(b[0] * sx)), int(round(b[1] * sy)),
                                        int(round(b[2] * sx)), int(round(b[3] * sy))), label))
        return results

    def iou(p, q):
        ix1, iy1 = max(p[0], q[0]), max(p[1], q[1])
        ix2, iy2 = min(p[2], q[2]), min(p[3], q[3])
        inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
        ua = (p[2]-p[0])*(p[3]-p[1]) + (q[2]-q[0])*(q[3]-q[1]) - inter
        return inter / ua if ua > 0 else 0.0

    def matches(ref, test):
        if len(ref) != len(test):
            return False
        used = [False] * len(test)
        for rb, _ in ref:
            best, bi = -1, 0.0
            for j, (tb, _) in enumerate(test):
                if not used[j]:
                    v = iou(rb, tb)
                    if v > bi:
                        best, bi = j, v
            if best < 0 or bi < 0.9:
                return False
            used[best] = True
        return True

    def redact(fr, ds):
        o = fr.copy(); h, w = fr.shape[:2]
        for (x1, y1, x2, y2), _ in ds:
            x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(w, x2), min(h, y2)
            if x2 <= x1 or y2 <= y1:
                continue
            if a.redact == "blue":
                o[y1:y2, x1:x2] = (255, 0, 0)
            else:
                o[y1:y2, x1:x2] = cv2.GaussianBlur(o[y1:y2, x1:x2], (31, 31), 0)
        return o

    cap = cv2.VideoCapture(a.video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    ow, oh = (H, W) if a.rotate in (90, 270) else (W, H)
    writer = cv2.VideoWriter(a.out, cv2.VideoWriter_fourcc(*"mp4v"), fps, (ow, oh))

    if dev == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    n = ff = lf = nf = nl = idx = 0
    buf = []
    use_batch = a.batch_size > 1

    with patch_instances(fields=PATCH_INSTANCES_FIELDS), torch.inference_mode():
        # Startup self-check: batched output must match the trusted per-frame path.
        if use_batch:
            probe, pc = [], cv2.VideoCapture(a.video)
            for _ in range(min(a.batch_size, 4)):
                ok, fr = pc.read()
                if not ok:
                    break
                probe.append(cv2.rotate(fr, rflag) if rflag is not None else fr)
            pc.release()
            try:
                if probe and all(matches(detect_one(f), b)
                                 for f, b in zip(probe, detect_batch(probe))):
                    print("self-check OK -- batched matches per-frame")
                else:
                    use_batch = False
                    print("self-check FAILED -- using per-frame")
            except Exception as e:
                use_batch = False
                print("batched path errored -- using per-frame:", repr(e)[:160])

        def flush():
            nonlocal n, ff, lf, nf, nl, buf
            if not buf:
                return
            dl = detect_batch(buf) if use_batch else [detect_one(f) for f in buf]
            for fr, ds in zip(buf, dl):
                labels = [d[1] for d in ds]
                ff += ("face" in labels); lf += ("licence plate" in labels)
                nf += labels.count("face"); nl += labels.count("licence plate")
                writer.write(redact(fr, ds))
                n += 1
                if a.progress and n % 100 == 0:
                    with open(a.progress, "w") as pf:
                        pf.write(str(n))
            buf = []

        while idx < a.end:
            ok, fr = cap.read()
            if not ok:
                break
            if idx < a.start:                    # fast-skip frames owned by another worker
                idx += 1
                continue
            buf.append(cv2.rotate(fr, rflag) if rflag is not None else fr)
            idx += 1
            if len(buf) >= a.batch_size:
                flush()
        flush()

    cap.release(); writer.release()
    el = time.perf_counter() - t0
    if a.progress:
        with open(a.progress, "w") as pf:
            pf.write(str(n))
    peak = round(torch.cuda.max_memory_allocated() / 1e9, 3) if dev == "cuda" else None
    with open(a.stats, "w") as sf:
        json.dump({"start": a.start, "end": a.end, "frames": n, "elapsed_s": round(el, 2),
                   "fps": round(n / el, 2) if el else None, "face_frames": ff,
                   "lp_frames": lf, "n_face": nf, "n_lp": nl, "peak_vram_gb": peak,
                   "batched": use_batch, "batch_size": a.batch_size, "out": a.out}, sf)
    print("worker", a.start, "-", a.end, "done:", n, "frames in", round(el, 1), "s",
          "(batched)" if use_batch else "(per-frame)")


if __name__ == "__main__":
    main()

## Step 6 -- run (N processes x batch) + stitch
Knobs at the top of the cell: `NUM_PROCS` x `BATCH_SIZE` is the total speedup. `MAX_FRAMES` defaults to a short smoke test -- set it to `None` for the whole clip. A live counter shows progress; the final JSON is your run record. Watch `peak_vram_gb_each` -- if a worker OOMs, lower `BATCH_SIZE` or `NUM_PROCS`.

In [ ]:
import sys, os, subprocess, json, time, glob

# ---- knobs -----------------------------------------------------------------
NUM_PROCS  = 2          # parallel processes sharing the GPU
BATCH_SIZE = 4          # frames batched per process (4 was the measured sweet spot)
ROTATE     = 0          # 0 / 90 / 180 / 270 -- pick from the Step 4 preview
REDACT     = "blur"     # "blur" (thesis default) or "blue" (solid fill)
MAX_FRAMES = 400        # quick smoke test; set None for the WHOLE clip
OUT_PATH   = "egoblur_anonymized.mp4"
WORKER     = "egoblur_worker.py"

# Native resolution = fit to the data (no rescale). Raise both (e.g. 800) to
# upscale for small/distant faces if face_coverage looks low.
MIN_SIZE, MAX_SIZE = min(W, H), max(W, H)

for f in glob.glob("_seg*"):
    os.remove(f)
total = N_TOTAL if not MAX_FRAMES else min(N_TOTAL, MAX_FRAMES)
bounds = [round(i * total / NUM_PROCS) for i in range(NUM_PROCS + 1)]

procs, logs, segs, statfs = [], [], [], []
t0 = time.perf_counter()
for i in range(NUM_PROCS):
    s = bounds[i]
    e = bounds[i + 1] if (i < NUM_PROCS - 1 or MAX_FRAMES) else 10**9   # last worker -> EOF
    seg, stf, prog, logf = f"_seg{i}.mp4", f"_seg{i}.json", f"_seg{i}.prog", f"_seg{i}.log"
    segs.append(seg); statfs.append(stf)
    cmd = [sys.executable, WORKER, "--video", VIDEO_PATH, "--out", seg, "--stats", stf,
           "--progress", prog, "--start", str(s), "--end", str(e),
           "--min-size", str(MIN_SIZE), "--max-size", str(MAX_SIZE),
           "--batch-size", str(BATCH_SIZE), "--rotate", str(ROTATE), "--redact", REDACT]
    if FACE_MODEL:
        cmd += ["--face-model", FACE_MODEL]
    if LP_MODEL:
        cmd += ["--lp-model", LP_MODEL]
    lh = open(logf, "w"); logs.append(lh)
    procs.append(subprocess.Popen(cmd, stdout=lh, stderr=subprocess.STDOUT))

print(f"launched {NUM_PROCS} workers x batch {BATCH_SIZE} over ~{total} frames "
      f"({MIN_SIZE}x{MAX_SIZE} native, FP32, redact={REDACT}, rotate={ROTATE})")
while any(p.poll() is None for p in procs):
    done = 0
    for i in range(NUM_PROCS):
        try:
            done += int(open(f"_seg{i}.prog").read() or "0")
        except Exception:
            pass
    print(f"  running... ~{done}/{total} frames  ({int(time.perf_counter() - t0)}s)",
          end="\r", flush=True)
    time.sleep(3)
print()
for lh in logs:
    lh.close()
elapsed = time.perf_counter() - t0

for i, p in enumerate(procs):
    if p.returncode != 0:
        print(f"--- worker {i} FAILED (rc={p.returncode}) ---")
        print(open(f"_seg{i}.log").read()[-2500:])
assert all(p.returncode == 0 for p in procs), "a worker failed -- see log above"

st = [json.load(open(f)) for f in statfs]
n = sum(s["frames"] for s in st)

# Stitch the non-empty segments in order; re-encode for a clean, portable mp4.
keep = [seg for seg, s in zip(segs, st) if s["frames"] > 0]
with open("_segs.txt", "w") as f:
    for seg in keep:
        f.write(f"file '{os.path.abspath(seg)}'\n")
subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", "_segs.txt",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", OUT_PATH],
               check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

STATS = {
    "video": VIDEO_PATH, "procs": NUM_PROCS, "batch_size": BATCH_SIZE, "precision": "fp32",
    "batched_each": [s.get("batched") for s in st],
    "min_size": MIN_SIZE, "max_size": MAX_SIZE, "rotate_deg": ROTATE, "redact": REDACT,
    "frames_processed": n, "wall_s": round(elapsed, 2),
    "fps_processing": round(n / elapsed, 2) if elapsed else None,
    "per_worker_fps": [s["fps"] for s in st], "capture_fps": round(fps_in, 1),
    "peak_vram_gb_each": [s["peak_vram_gb"] for s in st],
    "face_detections": sum(s["n_face"] for s in st),
    "plate_detections": sum(s["n_lp"] for s in st),
    "face_coverage": round(sum(s["face_frames"] for s in st) / n, 3) if n else None,
    "plate_coverage": round(sum(s["lp_frames"] for s in st) / n, 3) if n else None,
    "output": OUT_PATH,
}
print(json.dumps(STATS, indent=2))

## Step 7 -- preview + download
Samples a few frames from the anonymized output. Uncomment the last line to download the clip.

In [ ]:
import matplotlib.pyplot as plt
cap = cv2.VideoCapture(OUT_PATH)
m = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or STATS["frames_processed"]
idxs = [max(0, int(m * f)) for f in (0.1, 0.3, 0.5, 0.7, 0.9)]
fig, ax = plt.subplots(1, len(idxs), figsize=(18, 4))
for a, fi in zip(ax, idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, fi); ok, fr = cap.read()
    if ok:
        a.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); a.set_title(f"out frame {fi}")
    a.axis("off")
cap.release(); plt.tight_layout(); plt.show()
print("Anonymized video:", OUT_PATH)
# Download it:
# from google.colab import files; files.download(OUT_PATH)

---
### Notes
- **Throughput (RQ2):** for real timing set `MAX_FRAMES = None` and compare `wall_s` / `fps_processing`. The two levers stack but neither is free: processes share one GPU (expect ~1.3-1.8x from `NUM_PROCS=2`, bounded by the GPU and Colab's ~2 vCPUs), and batching past ~4 stops helping this two-stage detector. Sweep `NUM_PROCS` x `BATCH_SIZE` and keep the best.
- **`batched_each`** in the stats shows whether each worker actually batched or fell back to per-frame (a failed self-check).
- **Recall vs resolution (M1):** native size can miss small/distant faces. If `face_coverage` looks low *and orientation is correct*, raise `MIN_SIZE, MAX_SIZE` (e.g. 800).
- **No tracking:** EgoBlur is per-frame, so a one-frame miss leaks on that frame -- the gap the temporal metric measures (`planning/METHOD.md`).
- **Privacy:** keep the anonymized output; never commit original/unblurred frames or identifying media.